# 01 — Sorteio estratificado para avaliação humana

Este notebook seleciona **259 BDDs** para avaliação humana a partir dos arquivos JSON das gerações.

## Desenho da amostragem

- 5 modelos
- 3 técnicas (`zero-shot`, `one-shot`, `few-shot`)
- 15 combinações `modelo × técnica`
- 259 casos-fonte
- 10 execuções por combinação

Cada `case_id` aparece **exatamente uma vez** na avaliação humana.

A distribuição é balanceada:

- cada combinação `modelo × técnica` recebe **17 ou 18 casos**;
- cada modelo recebe **51 ou 52 casos**;
- cada técnica recebe **86 ou 87 casos**;
- as execuções 1–10 ficam globalmente balanceadas em **25 ou 26 ocorrências**;
- a `seed` é fixa para permitir reprodução exata do sorteio.

## Saídas

1. `chave_amostragem.json`  
   Arquivo do pesquisador. Guarda `model`, `technique`, `execution`, `generation_id`, caso original e BDD sorteado.

2. `avaliacao_humana.json`  
   Arquivo cego para preenchimento das notas de:
   - Estrutura
   - Semântica
   - Detalhes

A nota final **não é calculada neste notebook**.


In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import json
import random

## 1. Configurações

In [ ]:
# Pasta contendo os 15 arquivos JSON:
# 5 modelos × 3 técnicas
PASTA_JSON = Path("geracoes")

TOTAL_CASOS = 259
QUANTIDADE_MODELOS = 5
QUANTIDADE_TECNICAS = 3
QUANTIDADE_EXECUCOES = 10

# Seed fixa para reprodutibilidade
SEED = 20260826

ARQUIVO_CHAVE = Path("chave_amostragem.json")
ARQUIVO_AVALIACAO = Path("avaliacao_humana.json")

randomizador = random.Random(SEED)

## 2. Funções auxiliares

In [ ]:
def salvar_json(caminho, dados):
    with open(caminho, "w", encoding="utf-8") as arquivo:
        json.dump(
            dados,
            arquivo,
            ensure_ascii=False,
            indent=2
        )


def normalizar_texto(texto):
    if texto is None:
        return None
    return str(texto).strip()


def criar_quotas_balanceadas(modelos, tecnicas, total, rng):
    """
    Cria quotas para as combinações modelo × técnica.

    Para 259 casos e 15 combinações:
    - 11 combinações recebem 17
    - 4 combinações recebem 18

    Os quatro extras são distribuídos de forma que:
    - quatro modelos diferentes recebem +1;
    - entre as técnicas, uma recebe dois extras e as demais um extra.
    """
    celulas = [
        (modelo, tecnica)
        for modelo in modelos
        for tecnica in tecnicas
    ]

    quota_base = total // len(celulas)
    resto = total % len(celulas)

    quotas = {
        celula: quota_base
        for celula in celulas
    }

    if resto == 0:
        return quotas

    # Para este desenho: 4 extras, 5 modelos e 3 técnicas.
    if resto <= len(modelos):
        modelos_extra = rng.sample(modelos, resto)

        tecnicas_extra = []

        # Distribuição mais equilibrada possível das técnicas.
        while len(tecnicas_extra) < resto:
            bloco = list(tecnicas)
            rng.shuffle(bloco)
            tecnicas_extra.extend(bloco)

        tecnicas_extra = tecnicas_extra[:resto]
        rng.shuffle(tecnicas_extra)

        for modelo, tecnica in zip(modelos_extra, tecnicas_extra):
            quotas[(modelo, tecnica)] += 1
    else:
        # Fallback genérico.
        celulas_extra = rng.sample(celulas, resto)
        for celula in celulas_extra:
            quotas[celula] += 1

    return quotas


def criar_plano_execucoes(quotas, quantidade_execucoes, rng, max_tentativas=5000):
    """
    Cria um plano globalmente balanceado para as execuções.

    Cada combinação modelo × técnica recebe inicialmente uma ocorrência
    de cada execução 1..10. As ocorrências adicionais são distribuídas
    de modo que, no total dos 259 casos, cada execução apareça 25 ou 26 vezes.

    Retorna:
    - alvo_global: quantidade desejada de cada execução;
    - plano: lista de execuções a ser usada em cada estrato.
    """
    celulas = list(quotas.keys())
    quantidade_celulas = len(celulas)
    total = sum(quotas.values())

    base_global = total // quantidade_execucoes
    resto_global = total % quantidade_execucoes

    alvo_global = {
        execucao: base_global
        for execucao in range(1, quantidade_execucoes + 1)
    }

    execucoes_com_extra = rng.sample(
        list(alvo_global.keys()),
        resto_global
    )

    for execucao in execucoes_com_extra:
        alvo_global[execucao] += 1

    # Cada execução já aparece uma vez em cada uma das 15 células.
    alvos_repeticoes = {
        execucao: alvo_global[execucao] - quantidade_celulas
        for execucao in alvo_global
    }

    repeticoes_por_celula = {
        celula: quotas[celula] - quantidade_execucoes
        for celula in celulas
    }

    if sum(alvos_repeticoes.values()) != sum(repeticoes_por_celula.values()):
        raise RuntimeError("Inconsistência na montagem do plano de execuções.")

    for _ in range(max_tentativas):
        restantes = dict(alvos_repeticoes)
        repetidas_escolhidas = {}

        ordem_celulas = sorted(
            celulas,
            key=lambda c: (-repeticoes_por_celula[c], rng.random())
        )

        falhou = False

        for celula in ordem_celulas:
            quantidade_repetidas = repeticoes_por_celula[celula]

            candidatas = [
                execucao
                for execucao, restante in restantes.items()
                if restante > 0
            ]

            if len(candidatas) < quantidade_repetidas:
                falhou = True
                break

            # Prioriza as execuções que ainda precisam aparecer mais vezes.
            candidatas.sort(
                key=lambda e: (-restantes[e], rng.random())
            )

            escolhidas = candidatas[:quantidade_repetidas]
            repetidas_escolhidas[celula] = escolhidas

            for execucao in escolhidas:
                restantes[execucao] -= 1

        if falhou:
            continue

        if not all(valor == 0 for valor in restantes.values()):
            continue

        plano = {}

        for celula in celulas:
            execucoes = list(range(1, quantidade_execucoes + 1))
            execucoes.extend(repetidas_escolhidas[celula])
            rng.shuffle(execucoes)
            plano[celula] = execucoes

        return alvo_global, plano

    raise RuntimeError(
        "Não foi possível construir um plano balanceado de execuções."
    )

## 3. Carregar os JSONs de geração

In [ ]:
arquivos_json = sorted(PASTA_JSON.glob("*.json"))

if not arquivos_json:
    raise RuntimeError(
        f"Nenhum arquivo JSON encontrado em: {PASTA_JSON.resolve()}"
    )

dados_por_estrato = {}

for caminho in arquivos_json:
    with open(caminho, "r", encoding="utf-8") as arquivo:
        dados = json.load(arquivo)

    modelo = dados["model"]
    tecnica = dados["technique"]
    chave_estrato = (modelo, tecnica)

    if chave_estrato in dados_por_estrato:
        raise ValueError(
            f"Mais de um arquivo encontrado para {modelo} / {tecnica}"
        )

    casos = {}

    for caso in dados["cases"]:
        case_id = caso["case_id"]

        geracoes = {}

        for geracao in caso["generations"]:
            execucao = int(geracao["execution"])

            if execucao in geracoes:
                raise ValueError(
                    f"{caminho.name}: execução {execucao} duplicada em {case_id}"
                )

            geracoes[execucao] = {
                "generation_id": geracao["generation_id"],
                "gherkin": geracao["gherkin"]
            }

        execucoes_encontradas = set(geracoes.keys())
        execucoes_esperadas = set(
            range(1, QUANTIDADE_EXECUCOES + 1)
        )

        if execucoes_encontradas != execucoes_esperadas:
            raise ValueError(
                f"{caminho.name}: {case_id} não possui exatamente "
                f"as execuções 1..{QUANTIDADE_EXECUCOES}. "
                f"Encontradas: {sorted(execucoes_encontradas)}"
            )

        casos[case_id] = {
            "source_id": caso.get("source_id"),
            "source_line": caso.get("source_line"),
            "original_case": caso["original_case"],
            "generations": geracoes
        }

    dados_por_estrato[chave_estrato] = {
        "arquivo": caminho.name,
        "cases": casos
    }

print(f"Arquivos carregados: {len(arquivos_json)}")

Arquivos carregados: 15


## 4. Validar modelos, técnicas e casos-fonte

In [ ]:
modelos = sorted({
    modelo
    for modelo, tecnica in dados_por_estrato
})

tecnicas = sorted({
    tecnica
    for modelo, tecnica in dados_por_estrato
})

print("Modelos:")
for modelo in modelos:
    print(" -", modelo)

print("\nTécnicas:")
for tecnica in tecnicas:
    print(" -", tecnica)

if len(modelos) != QUANTIDADE_MODELOS:
    raise ValueError(
        f"Esperados {QUANTIDADE_MODELOS} modelos, "
        f"mas foram encontrados {len(modelos)}."
    )

if len(tecnicas) != QUANTIDADE_TECNICAS:
    raise ValueError(
        f"Esperadas {QUANTIDADE_TECNICAS} técnicas, "
        f"mas foram encontradas {len(tecnicas)}."
    )

quantidade_estratos_esperada = (
    QUANTIDADE_MODELOS * QUANTIDADE_TECNICAS
)

if len(dados_por_estrato) != quantidade_estratos_esperada:
    raise ValueError(
        f"Esperadas {quantidade_estratos_esperada} combinações "
        f"modelo × técnica, mas foram encontradas "
        f"{len(dados_por_estrato)}."
    )

primeiro_estrato = next(iter(dados_por_estrato))
casos_referencia = set(
    dados_por_estrato[primeiro_estrato]["cases"].keys()
)

if len(casos_referencia) != TOTAL_CASOS:
    raise ValueError(
        f"Esperados {TOTAL_CASOS} casos-fonte, "
        f"mas foram encontrados {len(casos_referencia)}."
    )

# Também valida se source_id e original_case são consistentes
# entre todos os modelos/técnicas.
referencia_detalhes = {
    case_id: (
        dados_por_estrato[primeiro_estrato]["cases"][case_id]["source_id"],
        normalizar_texto(
            dados_por_estrato[primeiro_estrato]["cases"][case_id]["original_case"]
        )
    )
    for case_id in casos_referencia
}

for chave_estrato, conteudo in dados_por_estrato.items():
    casos_atual = set(conteudo["cases"].keys())

    if casos_atual != casos_referencia:
        faltantes = sorted(casos_referencia - casos_atual)
        extras = sorted(casos_atual - casos_referencia)

        raise ValueError(
            f"Os casos não coincidem em {chave_estrato}.\n"
            f"Faltantes: {faltantes}\n"
            f"Extras: {extras}"
        )

    for case_id in casos_referencia:
        atual = conteudo["cases"][case_id]

        detalhe_atual = (
            atual["source_id"],
            normalizar_texto(atual["original_case"])
        )

        if detalhe_atual != referencia_detalhes[case_id]:
            raise ValueError(
                f"Inconsistência de caso-fonte em {case_id}, "
                f"estrato {chave_estrato}."
            )

print(f"\nValidação concluída: {TOTAL_CASOS} casos iguais nos 15 estratos.")

Modelos:
 - Qwen/Qwen3-8B
 - google/gemma-4-E4B-it
 - ibm-granite/granite-4.1-8b
 - meta-llama/Meta-Llama-3-8B-Instruct
 - mistralai/Mistral-7B-Instruct-v0.3

Técnicas:
 - few-shot
 - one-shot
 - zero-shot

Validação concluída: 259 casos iguais nos 15 estratos.


## 5. Criar quotas balanceadas por modelo × técnica

In [ ]:
quotas = criar_quotas_balanceadas(
    modelos=modelos,
    tecnicas=tecnicas,
    total=TOTAL_CASOS,
    rng=randomizador
)

print("Quotas modelo × técnica:\n")

for modelo in modelos:
    print(modelo)
    for tecnica in tecnicas:
        print(f"  {tecnica}: {quotas[(modelo, tecnica)]}")

print("\nTotal:", sum(quotas.values()))

Quotas modelo × técnica:

Qwen/Qwen3-8B
  few-shot: 18
  one-shot: 17
  zero-shot: 17
google/gemma-4-E4B-it
  few-shot: 17
  one-shot: 18
  zero-shot: 17
ibm-granite/granite-4.1-8b
  few-shot: 17
  one-shot: 17
  zero-shot: 17
meta-llama/Meta-Llama-3-8B-Instruct
  few-shot: 17
  one-shot: 17
  zero-shot: 18
mistralai/Mistral-7B-Instruct-v0.3
  few-shot: 17
  one-shot: 18
  zero-shot: 17

Total: 259


## 6. Criar plano globalmente balanceado das execuções

In [ ]:
alvo_execucoes, plano_execucoes = criar_plano_execucoes(
    quotas=quotas,
    quantidade_execucoes=QUANTIDADE_EXECUCOES,
    rng=randomizador
)

print("Distribuição-alvo das execuções:")
for execucao in range(1, QUANTIDADE_EXECUCOES + 1):
    print(f"Execução {execucao}: {alvo_execucoes[execucao]}")

Distribuição-alvo das execuções:
Execução 1: 26
Execução 2: 26
Execução 3: 26
Execução 4: 26
Execução 5: 26
Execução 6: 26
Execução 7: 26
Execução 8: 25
Execução 9: 26
Execução 10: 26


## 7. Sortear a combinação modelo × técnica para cada caso-fonte

In [ ]:
vagas = []

for estrato, quantidade in quotas.items():
    vagas.extend([estrato] * quantidade)

if len(vagas) != TOTAL_CASOS:
    raise RuntimeError("Quantidade incorreta de vagas.")

randomizador.shuffle(vagas)

case_ids = sorted(casos_referencia)
randomizador.shuffle(case_ids)

atribuicoes = defaultdict(list)

for case_id, estrato in zip(case_ids, vagas):
    atribuicoes[estrato].append(case_id)

if sum(len(ids) for ids in atribuicoes.values()) != TOTAL_CASOS:
    raise RuntimeError("Erro na atribuição dos casos aos estratos.")

## 8. Selecionar a execução exata de cada caso

In [ ]:
selecionados = []

for estrato, ids_casos in atribuicoes.items():
    modelo, tecnica = estrato

    ids_casos = list(ids_casos)
    randomizador.shuffle(ids_casos)

    execucoes = list(plano_execucoes[estrato])

    if len(ids_casos) != len(execucoes):
        raise RuntimeError(
            f"Quantidade de casos e execuções não coincide em {estrato}."
        )

    for case_id, execucao in zip(ids_casos, execucoes):
        caso = dados_por_estrato[estrato]["cases"][case_id]
        geracao = caso["generations"][execucao]

        selecionados.append({
            "case_id": case_id,
            "source_id": caso["source_id"],
            "source_line": caso["source_line"],
            "original_case": caso["original_case"],
            "model": modelo,
            "technique": tecnica,
            "execution": execucao,
            "generation_id": geracao["generation_id"],
            "gherkin": geracao["gherkin"]
        })

if len(selecionados) != TOTAL_CASOS:
    raise RuntimeError(
        f"Foram selecionados {len(selecionados)}, "
        f"mas eram esperados {TOTAL_CASOS}."
    )

# Cada caso-fonte deve aparecer exatamente uma vez.
ids_selecionados = [item["case_id"] for item in selecionados]

if len(set(ids_selecionados)) != TOTAL_CASOS:
    raise RuntimeError("Existem case_id repetidos na amostra.")

# Cada generation_id precisa ser único.
generation_ids = [item["generation_id"] for item in selecionados]

if len(set(generation_ids)) != TOTAL_CASOS:
    raise RuntimeError("Existem generation_id repetidos na amostra.")

print("Seleção concluída com sucesso.")

Seleção concluída com sucesso.


## 9. Validar o balanceamento final

In [ ]:
contagem_estratos = Counter(
    (item["model"], item["technique"])
    for item in selecionados
)

contagem_modelos = Counter(
    item["model"]
    for item in selecionados
)

contagem_tecnicas = Counter(
    item["technique"]
    for item in selecionados
)

contagem_execucoes = Counter(
    item["execution"]
    for item in selecionados
)

for estrato, esperado in quotas.items():
    encontrado = contagem_estratos[estrato]

    if encontrado != esperado:
        raise RuntimeError(
            f"Quota incorreta em {estrato}: "
            f"esperado {esperado}, encontrado {encontrado}."
        )

for execucao, esperado in alvo_execucoes.items():
    encontrado = contagem_execucoes[execucao]

    if encontrado != esperado:
        raise RuntimeError(
            f"Execução {execucao}: esperado {esperado}, "
            f"encontrado {encontrado}."
        )

print("Por modelo:")
for modelo in modelos:
    print(f"  {modelo}: {contagem_modelos[modelo]}")

print("\nPor técnica:")
for tecnica in tecnicas:
    print(f"  {tecnica}: {contagem_tecnicas[tecnica]}")

print("\nPor execução:")
for execucao in range(1, QUANTIDADE_EXECUCOES + 1):
    print(f"  Execução {execucao}: {contagem_execucoes[execucao]}")

Por modelo:
  Qwen/Qwen3-8B: 52
  google/gemma-4-E4B-it: 52
  ibm-granite/granite-4.1-8b: 51
  meta-llama/Meta-Llama-3-8B-Instruct: 52
  mistralai/Mistral-7B-Instruct-v0.3: 52

Por técnica:
  few-shot: 86
  one-shot: 87
  zero-shot: 86

Por execução:
  Execução 1: 26
  Execução 2: 26
  Execução 3: 26
  Execução 4: 26
  Execução 5: 26
  Execução 6: 26
  Execução 7: 26
  Execução 8: 25
  Execução 9: 26
  Execução 10: 26


## 10. Embaralhar ordem da avaliação e criar IDs cegos

In [ ]:
randomizador.shuffle(selecionados)

for indice, item in enumerate(selecionados, start=1):
    item["avaliacao_id"] = f"AV_{indice:03d}"

print("Exemplo de IDs:")
for item in selecionados[:5]:
    print(item["avaliacao_id"], "->", item["case_id"])

Exemplo de IDs:
AV_001 -> TC_559
AV_002 -> TC_7
AV_003 -> TC_341
AV_004 -> TC_848
AV_005 -> TC_431


## 11. Gerar `chave_amostragem.json`

In [ ]:
chave_amostragem = {
    "metadata": {
        "seed": SEED,
        "total_casos": TOTAL_CASOS,
        "quantidade_modelos": QUANTIDADE_MODELOS,
        "quantidade_tecnicas": QUANTIDADE_TECNICAS,
        "quantidade_execucoes": QUANTIDADE_EXECUCOES,
        "metodo": "amostragem balanceada por modelo x tecnica",
        "unidade_amostral": "uma geracao BDD por caso-fonte",
        "chave_para_metricas": "generation_id",
        "observacao_metricas": (
            "generation_id identifica exatamente caso, modelo, tecnica e execucao"
        )
    },
    "selecoes": []
}

for item in selecionados:
    chave_amostragem["selecoes"].append({
        "avaliacao_id": item["avaliacao_id"],
        "case_id": item["case_id"],
        "source_id": item["source_id"],
        "source_line": item["source_line"],
        "original_case": item["original_case"],
        "model": item["model"],
        "technique": item["technique"],
        "execution": item["execution"],
        "generation_id": item["generation_id"],
        "gherkin": item["gherkin"]
    })

salvar_json(
    ARQUIVO_CHAVE,
    chave_amostragem
)

print(f"Gerado: {ARQUIVO_CHAVE.resolve()}")

Gerado: /content/chave_amostragem.json


## 12. Gerar `avaliacao_humana.json`

In [ ]:
avaliacao_humana = {
    "metadata": {
        "total_casos": TOTAL_CASOS,
        "escala": {
            "minimo": 0,
            "maximo": 10
        },
        "criterios": {
            "estrutura": {
                "peso": 4,
                "descricao": (
                    "Avaliar clareza e adequacao estrutural do cenario BDD/Gherkin."
                )
            },
            "semantica": {
                "peso": 4,
                "descricao": (
                    "Avaliar se o BDD preserva corretamente a intencao "
                    "do caso de teste original."
                )
            },
            "detalhes": {
                "peso": 2,
                "descricao": (
                    "Avaliar a presenca de dados, condicoes e detalhes "
                    "relevantes para o comportamento testado."
                )
            }
        },
        "observacao": (
            "Preencher apenas estrutura, semantica e detalhes. "
            "A nota final sera calculada no Notebook 02."
        )
    },
    "avaliacoes": []
}

for item in selecionados:
    avaliacao_humana["avaliacoes"].append({
        "avaliacao_id": item["avaliacao_id"],
        "case_id": item["case_id"],
        "caso_original": item["original_case"],
        "bdd_gerado": item["gherkin"],
        "avaliacao": {
            "estrutura": None,
            "semantica": None,
            "detalhes": None
        }
    })

salvar_json(
    ARQUIVO_AVALIACAO,
    avaliacao_humana
)

print(f"Gerado: {ARQUIVO_AVALIACAO.resolve()}")

Gerado: /content/avaliacao_humana.json


## 13. Resumo final

In [ ]:
print("=" * 70)
print("AMOSTRAGEM CONCLUÍDA")
print("=" * 70)

print(f"Seed: {SEED}")
print(f"Total selecionado: {len(selecionados)}")
print(f"Case IDs únicos: {len(set(ids_selecionados))}")
print(f"Generation IDs únicos: {len(set(generation_ids))}")

print("\nModelo × técnica:")
for modelo in modelos:
    print(f"\n{modelo}")
    for tecnica in tecnicas:
        print(
            f"  {tecnica}: "
            f"{contagem_estratos[(modelo, tecnica)]}"
        )

print("\nExecuções:")
for execucao in range(1, QUANTIDADE_EXECUCOES + 1):
    print(
        f"  Execução {execucao}: "
        f"{contagem_execucoes[execucao]}"
    )

print("\nArquivos:")
print(f" - {ARQUIVO_CHAVE}")
print(f" - {ARQUIVO_AVALIACAO}")

AMOSTRAGEM CONCLUÍDA
Seed: 20260826
Total selecionado: 259
Case IDs únicos: 259
Generation IDs únicos: 259

Modelo × técnica:

Qwen/Qwen3-8B
  few-shot: 18
  one-shot: 17
  zero-shot: 17

google/gemma-4-E4B-it
  few-shot: 17
  one-shot: 18
  zero-shot: 17

ibm-granite/granite-4.1-8b
  few-shot: 17
  one-shot: 17
  zero-shot: 17

meta-llama/Meta-Llama-3-8B-Instruct
  few-shot: 17
  one-shot: 17
  zero-shot: 18

mistralai/Mistral-7B-Instruct-v0.3
  few-shot: 17
  one-shot: 18
  zero-shot: 17

Execuções:
  Execução 1: 26
  Execução 2: 26
  Execução 3: 26
  Execução 4: 26
  Execução 5: 26
  Execução 6: 26
  Execução 7: 26
  Execução 8: 25
  Execução 9: 26
  Execução 10: 26

Arquivos:
 - chave_amostragem.json
 - avaliacao_humana.json
